In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

import statsmodels.api as sm

from ISLP.models import ModelSpec as MS
from ISLP.models import poly
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_validate, KFold


# Cross Validating Classifier Problems

In [ ]:

np.random.seed(318)
n = 10**4
x1 = np.random.uniform(-0.5, 0.5, n)
x2 = np.random.uniform(0, 1, n)

cls = 1 - (x1 < 0.25 * np.sin(4 * np.pi * x2)).astype(int)
nonlinear_df = pd.DataFrame({"x1": x1, "x2": x2, "class": pd.Categorical(cls)})

fig, ax = plt.subplots()
for k, g in nonlinear_df.groupby("class"):
    g.plot.scatter("x1", "x2", s=10, alpha=0.5, label=f"Class {k}", c='class',cmap="viridis", colorbar=False, ax=ax)
ax.set_title("Nonlinear Decision Boundary")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.legend()


In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
design = MS(['x1', 'x2'], intercept=False) 
X = design.fit_transform(nonlinear_df)

logistic = LogisticRegression()
lda = LDA()
qda = QDA()
knn = KNeighborsClassifier(n_neighbors=3)

loocv = LeaveOneOut()

cv_results_logistic = cross_validate(logistic, X, nonlinear_df["class"], cv=loocv, return_train_score=True,scoring='accuracy')
cv_results_lda = cross_validate(lda, X, nonlinear_df["class"], cv=loocv, return_train_score=True,scoring='accuracy')
cv_results_qda = cross_validate(qda, X, nonlinear_df["class"], cv=loocv, return_train_score=True,scoring='accuracy')
cv_results_knn = cross_validate(knn, X, nonlinear_df["class"], cv=loocv, return_train_score=True,scoring='accuracy')


In [ ]:
print(1- cv_results_logistic['train_score'].mean(), 1- cv_results_logistic['test_score'].mean())
print(1- cv_results_lda['train_score'].mean(), 1- cv_results_lda['test_score'].mean())
print(1- cv_results_qda['train_score'].mean(), 1- cv_results_qda['test_score'].mean())
print(1- cv_results_knn['train_score'].mean(), 1- cv_results_knn['test_score'].mean())

In [ ]:
design = MS(['x1', 'x2'], intercept=False) 
X = design.fit_transform(nonlinear_df)

logistic = LogisticRegression()
lda = LDA()
qda = QDA()
knn = KNeighborsClassifier(n_neighbors=3)

kfold_cv = KFold(n_splits=10, shuffle=True, random_state=318)

cv_results_logistic = cross_validate(logistic, X, nonlinear_df["class"], cv=kfold_cv, return_train_score=True,scoring='accuracy')
cv_results_lda = cross_validate(lda, X, nonlinear_df["class"], cv=kfold_cv, return_train_score=True,scoring='accuracy')
cv_results_qda = cross_validate(qda, X, nonlinear_df["class"], cv=kfold_cv, return_train_score=True,scoring='accuracy')
cv_results_knn = cross_validate(knn, X, nonlinear_df["class"], cv=kfold_cv, return_train_score=True,scoring='accuracy')


In [ ]:
print(1- cv_results_logistic['train_score'].mean(), 1- cv_results_logistic['test_score'].mean())
print(1- cv_results_lda['train_score'].mean(), 1- cv_results_lda['test_score'].mean())
print(1- cv_results_qda['train_score'].mean(), 1- cv_results_qda['test_score'].mean())
print(1- cv_results_knn['train_score'].mean(), 1- cv_results_knn['test_score'].mean())

In [ ]:
neighbors = [1, 2, 3, 4, 5]
kfold_cv = KFold(n_splits=10, shuffle=True, random_state=318)

training_cv_results_knn = []
testing_cv_results_knn = []
for k in neighbors:
    knn = KNeighborsClassifier(n_neighbors=k)
    cv_results_knn = cross_validate(knn, X, nonlinear_df["class"], cv=kfold_cv, return_train_score=True,scoring='accuracy')
    training_cv_results_knn.append(1 - cv_results_knn['train_score'].mean())
    testing_cv_results_knn.append(1 - cv_results_knn['test_score'].mean())


In [ ]:
fig, ax = plt.subplots()
ax.plot(neighbors, training_cv_results_knn, label="Training Error")
ax.plot(neighbors, testing_cv_results_knn, label="Testing Error")
ax.set_xlabel("Number of Neighbors $k$")
ax.set_ylabel("Misclassification Rate")
ax.set_title("KNN Performance as a Function of $k$")
ax.legend()

# Bootstrap Methods


In [ ]:
from scipy.stats import bootstrap

# Reproducible random generator
rng = np.random.default_rng(42)

# Generate 100 samples from Exponential(1)
data = rng.exponential(scale=1.0, size=10000)

# Define statistic function
def estimate_kurtosis(data):
    return np.mean((data - np.mean(data))**4) / (np.mean((data - np.mean(data))**2)**2)

# Bootstrap
res = bootstrap(
    data=(data,),
    statistic=estimate_kurtosis,
    confidence_level=0.95,
    n_resamples=100,
    random_state=42
)

res.confidence_interval

In [ ]:
res.bootstrap_distribution